# Next step: step 8a, requests and returned data
**What it adds:** `simr.run(models, request)`, the first piece of the workflow above `sim`. A request (a plain dict)
names the simulations wanted and the data to hand back; `simr` runs only what's needed and returns it in memory.

| | What it is |
|---|---|
| `simr.run(models, request)` | Runs every model on every simulator entry; returns a `SimrResult` |
| `SimrResult.models` | The models that ran, `(N, 14)`: row m is model m |
| `SimrResult.simulators` | The simulator entries as `SimulatorConfig`s, every default filled in: entry s is simulator s |
| `SimrResult.seed` | The run's seed: the request's, or a random one when none was given |
| `SimrResult.data` | One entry per requested name: a numpy array of cells indexed `[model, simulator]` |
| `simr.SIM_DATA` | The names the Simulator stage can hand back: `sim.particles`, `sim.unperturbed`, `sim.orbit`, `sim.timing` |
| `simr.model_seed_for(seed, m)` | A model's seed: `SeedSequence([seed, m])`, shared by all simulator entries on that model |

**How it ties in:** `sim` runs one simulation; `simr` decides which ones and collects them. Later steps add saving
(8b), failure handling (9), the run folder and reuse (10), the config file (11) and pools (12). Scores will sit beside
`data` as `result.scores` once images and fitness exist. Simulated arrays become `float32` here, not in `sim`, so
`sim` still returns exactly what SPAM made.

## What changed
**New, `src/galactic_tidal_constraints/simr/simr.py`.** The names `simr` can hand back, each a small function on
top of `sim`:
```python
def _particles(params, config, seed):
    return sim.simulate_particles(params, config, seed=seed).astype(np.float32)
...
SIM_DATA = {
    "sim.particles": _particles,
    "sim.unperturbed": _unperturbed,
    "sim.orbit": _orbit,
    "sim.timing": _timing,
}
```
The result, a plain container that describes itself: what ran, with what, and what came back (later: `scores`,
`status`):
```python
@dataclass
class SimrResult:
    models: np.ndarray                 # (N, 14): row m is model m in data
    simulators: list[SimulatorConfig]  # entry s is simulator s in data
    seed: int
    data: dict[str, np.ndarray]
```
The loop. Only the requested names are computed, and every simulator entry on a model gets the same seed:
```python
def run(models, request):
    models = _check_models(models)
    seed, wanted, configs = _read_request(request)

    data = {name: np.empty((len(models), len(configs)), dtype=object) for name in wanted}
    for m, params in enumerate(models):
        model_seed = model_seed_for(seed, m)
        for s, config in enumerate(configs):
            for name in wanted:
                data[name][m, s] = SIM_DATA[name](params, config, model_seed)
    return SimrResult(models=models, simulators=configs, seed=seed, data=data)

def model_seed_for(run_seed, model_index):
    return int(np.random.SeedSequence([run_seed, model_index]).generate_state(1)[0])
```
`_read_request` rejects unknown keys and names, and builds a `SimulatorConfig` from each simulator dict.
`run.return` defaults to `[]`. With no `run.seed`, a random one is drawn and reported as `result.seed`:
```python
seed = run_section.get("seed")
if seed is None:
    seed = np.random.SeedSequence().entropy  # random, from the operating system
```

**Tests:** new `tests/test_simr.py`, section "Step 8a".
**Decisions** (`PLAN.md`): `run.return` / `run.save` lists of `stage.item` names; `float32` after simulating.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim, simr

# Line 3 of JSPAM's archive/data.txt: its secondary passes inside r = 2, where friction acts, so lnl matters.
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
models = np.stack([params, heavier])   # (2, 14): the model, and the same with mass2 doubled
N = 1500

## 1. A request
Two models, two simulator entries (only `lnl` differs), and three names to hand back:

In [ ]:
request = {
    "run": {"seed": 3, "return": ["sim.particles", "sim.orbit", "sim.timing"]},
    "simulator": [{"n1": N, "n2": N, "lnl": 0.0025}, {"n1": N, "n2": N, "lnl": 0.001}],
}
start = time.perf_counter()
result = simr.run(models, request)
print(f"{time.perf_counter() - start:.1f} s")
print(type(result).__name__, "with names:", list(result.data), " seed:", result.seed)

The result says what ran: `models` (row m) and `simulators` (entry s), with every default filled in, not only the
keys the request gave:

In [ ]:
print("models:", result.models.shape, " mass2 of each:", result.models[:, sim.PARAM_INDEX["mass2"]])
for s, config in enumerate(result.simulators):
    print(f"simulator {s}: {config}")

## 2. Cells, indexed `[model, simulator]`
Each name is an array of cells. A cell holds that run's own array, so orbits of different lengths fit side by side:

In [ ]:
for name, cells in result.data.items():
    print(f"{name}: {cells.shape}")
for m in range(len(result.models)):
    for s in range(len(result.simulators)):
        orbit = result.data["sim.orbit"][m, s]
        particles = result.data["sim.particles"][m, s]
        t_start = result.data["sim.timing"][m, s]["t_start"]
        print(f"  [{m}, {s}]  particles {particles.shape} {particles.dtype}   orbit {orbit.shape} {orbit.dtype}   t_start {t_start:.3f}")

Both models on both entries. Each row is one model, each column one simulator entry:

In [ ]:
n_models, n_sims = len(result.models), len(result.simulators)
fig, axes = plt.subplots(n_models, n_sims, figsize=(6 * n_sims, 6 * n_models), sharex=True, sharey=True, squeeze=False)
for m, model in enumerate(result.models):
    for s, config in enumerate(result.simulators):
        ax, p, orbit = axes[m, s], result.data["sim.particles"][m, s], result.data["sim.orbit"][m, s]
        n1 = config.n1
        ax.scatter(p[:n1, 0], p[:n1, 1], s=1, label="primary disk")
        ax.scatter(p[n1:-1, 0], p[n1:-1, 1], s=1, label="secondary disk")
        ax.plot(orbit[:, 0], orbit[:, 1], color="k", lw=1, label="secondary's orbit")
        mass2 = model[sim.PARAM_INDEX["mass2"]]
        ax.set_title(f"[{m}, {s}]  mass2 {mass2:.3g}, lnl {config.lnl:g}: starts at t = {orbit[0, 6]:.2f}")
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect("equal")
axes[0, 0].legend(loc="upper left", markerscale=5)
plt.show()

## 3. Only what's requested runs
Asking only for the orbit never runs particles, so it takes a fraction of the time:

In [ ]:
for names in (["sim.orbit"], ["sim.timing"], ["sim.unperturbed"], ["sim.particles"]):
    start = time.perf_counter()
    simr.run(models, {**request, "run": {"seed": 3, "return": names}})
    print(f"{str(names):22s} {time.perf_counter() - start:6.2f} s for 2 models x 2 entries")

## 4. Seeds
All entries on one model start from the same particles, so the difference between the two columns above comes from
`lnl`, not from different disks. A model's particles also don't depend on which other models are in the run:

In [ ]:
starts = simr.run(models, {**request, "run": {"seed": 3, "return": ["sim.unperturbed"]}}).data["sim.unperturbed"]
print("model 0, entries 0 and 1 start identical:", np.array_equal(starts[0, 0], starts[0, 1]))
print("models 0 and 1 start differently:        ", not np.array_equal(starts[0, 0][:result.simulators[0].n1], starts[1, 0][:result.simulators[0].n1]))

alone = simr.run(models[:1], {**request, "run": {"seed": 3, "return": ["sim.particles"]}}).data["sim.particles"]
print("model 0 alone = model 0 in the pair:     ", np.array_equal(alone[0, 0], result.data["sim.particles"][0, 0]))
print("model 0's seed:", simr.model_seed_for(result.seed, 0))

With no `run.seed`, each run draws its own. The result reports it, so passing it back repeats the run exactly:

In [ ]:
no_seed = {"run": {"return": ["sim.unperturbed"]}, "simulator": {"n1": N, "n2": N}}
first, second = simr.run(models[:1], no_seed), simr.run(models[:1], no_seed)
print("first seed: ", first.seed)
print("second seed:", second.seed)
print("different particles:", not np.array_equal(first.data["sim.unperturbed"][0, 0], second.data["sim.unperturbed"][0, 0]))
again = simr.run(models[:1], {**no_seed, "run": {"seed": first.seed, "return": ["sim.unperturbed"]}})
print("first repeated exactly:", np.array_equal(again.data["sim.unperturbed"][0, 0], first.data["sim.unperturbed"][0, 0]))

## 5. What `float32` costs
The largest change from rounding the particle run to `float32`, against the same run in `float64`:

In [ ]:
# model 0 on simulator 0, rerun directly with sim from what the result recorded
full = sim.simulate_particles(result.models[0], result.simulators[0], seed=simr.model_seed_for(result.seed, 0))
rounded = result.data["sim.particles"][0, 0]
print("same values after rounding:", np.array_equal(full.astype(np.float32), rounded))
print("largest position change:", np.abs(rounded[:, :3] - full[:, :3]).max())
print("largest relative change:", np.max(np.abs(rounded - full) / np.maximum(np.abs(full), 1e-30)))
print(f"memory: {full.nbytes / 1e3:.0f} kB -> {rounded.nbytes / 1e3:.0f} kB")

## 6. Bad requests name the problem
Mistakes are caught before any simulation runs:

In [ ]:
bad_requests = [
    {"runn": {}},
    {"run": {"return": ["sim.orbits"]}},
    {"run": {"return": "sim.orbit"}},
    {"run": {"save": ["sim.orbit"]}},
    {"run": {"seed": 1.5}},
    {"simulator": [{"n1": 10}, {"n_1": 10}]},
    {"simulator": [{"lnl": -1}]},
]
for bad in bad_requests:
    try:
        simr.run(models, bad)
    except ValueError as error:
        print(f"{str(bad):42s} -> {error}")